# 00 · BigQuery data health check

Runs in the BigQuery sandbox (1 TiB of free queries per month).

1. Freshness and gaps of `bigquery-public-data.crypto_bitcoin`
2. Data volume: rows, bytes, partitioning and clustering
3. Address and script fields of P2PK and Taproot outputs
4. Dry-run cost of the intermediate tables
5. Sample blocks reconciled against mempool.space

Every query is dry-run first and refused if it would scan more than `MAX_GB_PER_QUERY`.

In [ ]:
# Configuration
PROJECT_ID = ""              # your GCP project ID
MAX_GB_PER_QUERY = 200       # maximum scan per query (GB); larger queries are refused
PRICE_PER_TIB_USD = 6.25     # on-demand query price
FREE_TIB_PER_MONTH = 1.0     # free monthly allowance (sandbox and free tier)

assert PROJECT_ID, "Set PROJECT_ID first"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # when running locally, use: gcloud auth application-default login

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
PUB = "bigquery-public-data.crypto_bitcoin"
print("Project:", client.project)

In [ ]:
# Helpers
import json, time, datetime as dt
import requests

REPORT = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(), "project": PROJECT_ID}
STATE = {"scanned_bytes": 0.0}

def dry_run(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    gb = job.total_bytes_processed / 2**30
    return gb, gb / 1024 * PRICE_PER_TIB_USD

def run(sql, label="", max_gb=None):
    max_gb = MAX_GB_PER_QUERY if max_gb is None else max_gb
    gb, usd = dry_run(sql)
    print(f"[{label}] estimated scan {gb:,.2f} GB ~ ${usd:,.2f}")
    if gb > max_gb:
        raise RuntimeError(f"[{label}] scan of {gb:,.1f} GB exceeds the limit of {max_gb} GB; stopped")
    df = client.query(sql).to_dataframe()
    STATE["scanned_bytes"] += gb * 2**30
    print(f"[{label}] done. Scanned so far in this notebook: {STATE['scanned_bytes']/2**40:.4f} TiB "
          f"(free allowance {FREE_TIB_PER_MONTH} TiB/month)")
    return df

## Check 0 · Table metadata

Rows, bytes, partitioning and clustering fields (metadata only, no query).

In [ ]:
rows = []
PART = {}
for t in ["blocks", "transactions", "inputs", "outputs"]:
    tb = client.get_table(f"{PUB}.{t}")
    part = tb.time_partitioning.field if tb.time_partitioning else None
    PART[t] = part
    rows.append({
        "table": t,
        "rows": tb.num_rows,
        "TiB": round(tb.num_bytes / 2**40, 3),
        "partition_field": part,
        "clustering": tb.clustering_fields,
        "modified": tb.modified.isoformat() if tb.modified else None,
    })
import pandas as pd
meta = pd.DataFrame(rows)
REPORT["tables"] = rows
meta

## Check 1 · Freshness and gaps

Queries on the `blocks` table.

In [ ]:
sql = f"""
SELECT MIN(timestamp) AS min_ts, MAX(timestamp) AS max_ts,
       MAX(number) AS max_block, COUNT(*) AS n_blocks
FROM `{PUB}.blocks`
"""
fresh = run(sql, "freshness")
display(fresh)

max_ts = pd.to_datetime(fresh.loc[0, "max_ts"], utc=True)
lag_days = (pd.Timestamp.utcnow() - max_ts).total_seconds() / 86400
print(f"Latest block {int(fresh.loc[0,'max_block']):,}, {lag_days:.1f} days behind")
REPORT["freshness"] = {
    "max_block": int(fresh.loc[0, "max_block"]),
    "max_ts": str(fresh.loc[0, "max_ts"]),
    "lag_days": round(lag_days, 2),
    "n_blocks": int(fresh.loc[0, "n_blocks"]),
}

In [ ]:
sql = f"""
SELECT DATE(timestamp) AS d, COUNT(*) AS n
FROM `{PUB}.blocks`
WHERE timestamp >= TIMESTAMP("2021-01-01")
GROUP BY d ORDER BY d
"""
daily = run(sql, "daily_blocks")

full = pd.date_range(daily["d"].min(), daily["d"].max(), freq="D").date
missing = sorted(set(full) - set(daily["d"]))
thin = daily[daily["n"] < 100]
print("Days with no blocks:", len(missing))
print("Days with fewer than 100 blocks:", len(thin))
display(thin.tail(20))
REPORT["gaps"] = {
    "missing_days": [str(x) for x in missing[:50]],
    "n_missing_days": len(missing),
    "n_thin_days": int(len(thin)),
    "thin_days_tail": [{"d": str(r.d), "n": int(r.n)} for r in thin.tail(20).itertuples()],
}

## Check 2 · Script-type distribution

Full history if the dry-run scan is within the limit, otherwise the last three years.

In [ ]:
part_o = PART.get("outputs")
where_recent = f'WHERE {part_o} >= DATE("2023-01-01")' if part_o else 'WHERE block_timestamp >= TIMESTAMP("2023-01-01")'

sql_all = f"""
SELECT type,
       COUNT(*) AS n_outputs,
       ROUND(SUM(value)/1e8, 2) AS btc_created,
       MIN(block_number) AS first_block,
       MAX(block_number) AS last_block
FROM `{PUB}.outputs`
GROUP BY type ORDER BY n_outputs DESC
"""
gb, usd = dry_run(sql_all)
print(f"Full-history query: estimated scan {gb:,.1f} GB ~ ${usd:,.2f}")

if gb <= MAX_GB_PER_QUERY:
    types = run(sql_all, "script_types_all")
    REPORT["script_types_scope"] = "all_history"
else:
    sql_recent = sql_all.replace("FROM `" + PUB + ".outputs`", "FROM `" + PUB + ".outputs`\n" + where_recent)
    types = run(sql_recent, "script_types_recent")
    REPORT["script_types_scope"] = "since_2023"

display(types)
REPORT["script_types"] = types.to_dict(orient="records")

## Check 3 · P2PK and Taproot fields

- `pubkey` outputs: whether `addresses` is filled, and whether the public key can be read from `script_hex`
- `witness_v1_taproot` addresses start with bc1p
- redeem script visible in `script_asm` of P2SH inputs

In [ ]:
def one_month(table, cond, cols, month, limit=5):
    part = PART.get(table)
    w = f'{part} = DATE("{month}")' if part else f'DATE(block_timestamp) BETWEEN DATE("{month}") AND DATE_ADD(DATE("{month}"), INTERVAL 31 DAY)'
    return f"""
SELECT {cols}
FROM `{PUB}.{table}`
WHERE {w} AND {cond}
LIMIT {limit}
"""

samples = {}
samples["p2pk"] = run(one_month("outputs", "type = 'pubkey'",
                                "type, addresses, script_hex, value, block_number", "2010-01-01"), "sample_p2pk")
samples["taproot"] = run(one_month("outputs", "type = 'witness_v1_taproot'",
                                   "type, addresses, script_hex, value, block_number", "2023-06-01"), "sample_taproot")
samples["multisig"] = run(one_month("outputs", "type = 'multisig'",
                                    "type, addresses, script_hex, value, block_number", "2014-06-01"), "sample_multisig")
samples["p2sh_input"] = run(one_month("inputs", "type = 'scripthash'",
                                      "type, addresses, script_asm, value, block_number", "2019-06-01"), "sample_p2sh_input")

for k, v in samples.items():
    print("=" * 20, k)
    display(v)

REPORT["samples"] = {k: v.astype(str).to_dict(orient="records") for k, v in samples.items()}

## Check 4 · Build cost of the intermediate tables (dry run)

Step 01 scans each public table once; later notebooks read only the project tables.

In [ ]:
build_sql = {
"outputs_slim (without script_hex)": f"""
SELECT transaction_hash, index, block_number, block_timestamp, value, type,
       addresses[SAFE_OFFSET(0)] AS address
FROM `{PUB}.outputs`
""",
"outputs script_hex, separate pass": f"""
SELECT transaction_hash, index, script_hex
FROM `{PUB}.outputs`
""",
"inputs_slim (without script_asm)": f"""
SELECT transaction_hash, index, block_number, block_timestamp,
       spent_transaction_hash, spent_output_index, value, type,
       addresses[SAFE_OFFSET(0)] AS address
FROM `{PUB}.inputs`
""",
"inputs script_asm, separate pass": f"""
SELECT transaction_hash, index, script_asm
FROM `{PUB}.inputs`
""",
"utxo_life join": f"""
SELECT COUNT(*) AS n
FROM `{PUB}.outputs` o
LEFT JOIN `{PUB}.inputs` i
  ON i.spent_transaction_hash = o.transaction_hash
 AND i.spent_output_index = o.index
""",
"key_activity aggregation": f"""
SELECT address, MIN(block_timestamp) AS first_seen, MAX(block_timestamp) AS last_seen, COUNT(*) AS n
FROM (
  SELECT addresses[SAFE_OFFSET(0)] AS address, block_timestamp FROM `{PUB}.outputs`
  UNION ALL
  SELECT addresses[SAFE_OFFSET(0)] AS address, block_timestamp FROM `{PUB}.inputs`
)
GROUP BY address
""",
}

est, total_gb = [], 0.0
for name, sql in build_sql.items():
    gb, usd = dry_run(sql)
    total_gb += gb
    est.append({"query": name, "GB": round(gb, 1), "USD": round(usd, 2)})
    print(f"{name:38s} {gb:9,.1f} GB  ${usd:7,.2f}")

print("-" * 70)
print(f"{'Total':38s} {total_gb:9,.1f} GB  ${total_gb/1024*PRICE_PER_TIB_USD:7,.2f}")
print(f"= {total_gb/1024:.2f} TiB; free sandbox allowance {FREE_TIB_PER_MONTH} TiB per month")
REPORT["build_estimate"] = {"items": est, "total_gb": round(total_gb, 1),
                            "total_usd": round(total_gb/1024*PRICE_PER_TIB_USD, 2)}

## Check 5 · Sample blocks vs mempool.space

Five blocks from different years: hash, transaction count and timestamp.

In [ ]:
max_block = REPORT["freshness"]["max_block"]
heights = [150000, 350000, 550000, 750000, max_block - 1000]
recon = []

for h in heights:
    try:
        bh = requests.get(f"https://mempool.space/api/block-height/{h}", timeout=30).text.strip()
        blk = requests.get(f"https://mempool.space/api/block/{bh}", timeout=30).json()
    except Exception as e:
        print("mempool.space request failed:", e); break
    time.sleep(1)

    sql = f"""
    SELECT `hash` AS block_hash, number, timestamp, transaction_count
    FROM `{PUB}.blocks` WHERE number = {h}
    """
    bq = run(sql, f"block_{h}")
    ok_hash = (bq.loc[0, "block_hash"] == bh)
    ok_tx = int(bq.loc[0, "transaction_count"]) == int(blk["tx_count"])
    recon.append({"height": h, "hash_match": bool(ok_hash),
                  "bq_tx": int(bq.loc[0, "transaction_count"]), "mempool_tx": int(blk["tx_count"]),
                  "tx_match": bool(ok_tx)})
    print(recon[-1])

REPORT["reconciliation"] = recon

## Summary report

`healthcheck_report.json` records data freshness, gaps, the build-cost estimate and the block reconciliation.

In [ ]:
REPORT["scanned_tib_this_run"] = round(STATE["scanned_bytes"] / 2**40, 4)

print("=" * 60)
print("Data freshness :", REPORT["freshness"]["lag_days"], "days behind")
print("Missing days   :", REPORT["gaps"]["n_missing_days"])
print("Build estimate :", REPORT["build_estimate"]["total_gb"], "GB ~ $",
      REPORT["build_estimate"]["total_usd"])
print("Reconciliation :", REPORT.get("reconciliation"))
print("Scanned (run)  :", REPORT["scanned_tib_this_run"], "TiB")
print("=" * 60)

with open("healthcheck_report.json", "w", encoding="utf-8") as f:
    json.dump(REPORT, f, ensure_ascii=False, indent=2, default=str)

try:
    from google.colab import files
    files.download("healthcheck_report.json")
except Exception:
    print("Report saved to healthcheck_report.json")